# Experiment 8 (LLM) — Span Identification: evaluation

The LLM run script outputs predicted span **text**, but the official SemEval SI
scorer works on **character offsets**. This notebook bridges the two:

1. Run the model first: `python llm_span.py` (writes `results.json`).
2. Run this notebook — it locates each predicted/gold span string inside the
   article text, writes `.labels` files, and scores them with
   `../task-SI_scorer.py` (partial-overlap precision / recall / F1).


In [ ]:
import json
import subprocess

RESULTS_JSON = "results.json"       # produced by llm_span.py
SCORER       = "../task-SI_scorer.py"


In [ ]:
def find_offsets(text, span):
    """All (start, end) character ranges where `span` occurs in `text`."""
    offsets = []
    start = 0
    span = (span or "").strip()
    if not span:
        return offsets
    while True:
        i = text.find(span, start)
        if i < 0:
            break
        offsets.append((i, i + len(span)))
        start = i + len(span)
    return offsets


def merge(spans):
    """Merge overlapping/adjacent spans so the scorer gets clean, non-overlapping spans."""
    spans = sorted(spans)
    merged = []
    for s, e in spans:
        if merged and s <= merged[-1][1]:
            merged[-1] = (merged[-1][0], max(merged[-1][1], e))
        else:
            merged.append((s, e))
    return merged


In [ ]:
results = json.load(open(RESULTS_JSON))

gold_rows, pred_rows = [], []
for r in results:
    aid = str(r["article_id"])
    text = r["article_text"]

    gold = []
    for s in r.get("true_spans", []):
        gold += find_offsets(text, s)

    pred = []
    for s in r.get("predicted_spans", []):
        pred += find_offsets(text, s)

    for s, e in merge(gold):
        gold_rows.append((aid, s, e))
    for s, e in merge(pred):
        pred_rows.append((aid, s, e))

# The official scorer requires the submission's articles to be a subset of the
# reference's. Articles with no gold spans never appear in the gold file, so
# drop predictions for them to keep the article sets aligned.
gold_ids = {aid for aid, _, _ in gold_rows}
pred_rows = [row for row in pred_rows if row[0] in gold_ids]

print(f"articles: {len(gold_ids)} | gold spans: {len(gold_rows)} | predicted spans: {len(pred_rows)}")


In [ ]:
def write_labels(rows, path):
    with open(path, "w") as f:
        for aid, s, e in rows:
            f.write(f"{aid}\t{s}\t{e}\n")

write_labels(gold_rows, "llm_gold.labels")
write_labels(pred_rows, "llm_pred.labels")

result = subprocess.run(
    ["python", SCORER, "-s", "llm_pred.labels", "-r", "llm_gold.labels"],
    capture_output=True, text=True,
)
print(result.stdout)
print(result.stderr)
